In [ ]:
!pip install -q ragas==0.4.3 langchain==0.3.30 langchain-community==0.3.31 langchain-openai==0.3.35 rapidfuzz python-dotenv sentence-transformers pandas langchain-text-splitters
print("[✓] Tüm kütüphaneler başarıyla kuruldu.")

In [ ]:
!cp -r /kaggle/input/datasets/""/rag-data/rag_data .
!mkdir outputs

In [ ]:
import os
import sys
import datetime
import time
import asyncio
import hashlib
import pandas as pd

from openai import OpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.messages import BaseMessage, AIMessage

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.run_config import RunConfig
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

# Senin özel Azure Responses API Yapın
class CustomAzureResponsesModel(BaseChatModel):
    api_key: str = ""
    base_url: str = ""
    model_name: str = "gpt-4.1-mini"
    delay: float = 1.0

    def _generate(self, messages: list[BaseMessage], stop: list[str] | None = None, **kwargs) -> ChatResult:
        time.sleep(self.delay)
        prompt_text = "\n".join([msg.content for msg in messages])

        client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        response = client.responses.create(model=self.model_name, input=prompt_text)

        output_text = getattr(response, "output_text", str(response))
        message = AIMessage(content=output_text)
        return ChatResult(generations=[ChatGeneration(message=message)])

    async def _agenerate(self, messages: list[BaseMessage], stop: list[str] | None = None, **kwargs) -> ChatResult:
        await asyncio.sleep(self.delay)
        prompt_text = "\n".join([msg.content for msg in messages])

        client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        response = client.responses.create(model=self.model_name, input=prompt_text)

        output_text = getattr(response, "output_text", str(response))
        message = AIMessage(content=output_text)
        return ChatResult(generations=[ChatGeneration(message=message)])

    @property
    def _llm_type(self) -> str:
        return "custom-azure-responses-openai"

# Kaggle Ortam Parametreleri
TESTSET_SIZE = 100
MAX_WORKERS = 4
RAG_DATA_DIR = "/kaggle/working/rag_data"
OUTPUT_DIR = "/kaggle/working/outputs"
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 300

print("[✓] İstemci ve sabitler hazır.")

In [ ]:
def load_and_chunk_documents(data_dir):
    if not os.path.exists(data_dir):
        raise FileNotFoundError(f"Klasör bulunamadı: {data_dir}")
        
    print(f"[*] '{data_dir}' klasöründeki .md dosyaları yükleniyor...")
    loader = DirectoryLoader(data_dir, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    raw_docs = loader.load()
    if not raw_docs:
        raise ValueError(f"'{data_dir}' klasörü altında yüklenecek .md belgesi bulunamadı.")
        
    headers_to_split_on = [("#", "header_1"), ("##", "header_2"), ("###", "header_3"), ("####", "header_4")]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

    chunked_docs = []
    for raw_doc in raw_docs:
        header_docs = markdown_splitter.split_text(raw_doc.page_content)
        split_docs = text_splitter.split_documents(header_docs)
        source_name = os.path.basename(raw_doc.metadata.get("source", "unknown.md"))
        
        for doc in split_docs:
            doc.metadata["filename"] = source_name
            unique_string = source_name + doc.page_content
            doc.metadata["chunk_id"] = hashlib.md5(unique_string.encode('utf-8')).hexdigest()
            chunked_docs.append(doc)
            
    print(f"[✓] {len(raw_docs)} adet ham belge, {len(chunked_docs)} adet kusursuz parçaya dönüştürüldü.")
    return chunked_docs

# Fonksiyonu çalıştırıp belgeleri belleğe alalım
docs = load_and_chunk_documents(RAG_DATA_DIR)

In [ ]:
def init_ragas_generator():
    print("[*] Özel Azure Responses API istemcisi sarmalanıyor...")
    custom_llm = CustomAzureResponsesModel(model_name="gpt-4.1-mini", delay=1.0)
    
    print("[*] BAAI/bge-m3 embedding modeli yükleniyor...")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cuda" if os.system("nvidia-smi > /dev/null 2>&1") == 0 else "cpu"}
    )
    
    wrapped_llm = LangchainLLMWrapper(custom_llm)
    wrapped_embeddings = LangchainEmbeddingsWrapper(embeddings)
    return TestsetGenerator(llm=wrapped_llm, embedding_model=wrapped_embeddings), wrapped_llm

generator, wrapped_llm = init_ragas_generator()
print("[✓] Ragas Generator kullanıma hazır.")

In [ ]:
print("="*60)
print(f"      RAGAS TEST SETİ ÜRETİMİ BAŞLATILIYOR ({TESTSET_SIZE} SORU)")
print("="*60)

run_config = RunConfig(max_workers=MAX_WORKERS)

try:
    # Ragas 0.4.3 sürümüne tam uyumlu çağrı biçimi
    testset = generator.generate_with_langchain_docs(
        documents=docs, 
        testset_size=TESTSET_SIZE, 
        run_config=run_config
    )
    df = testset.to_pandas()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    ref_col = "reference_contexts" if "reference_contexts" in df.columns else "contexts" if "contexts" in df.columns else None
    if not ref_col:
        raise ValueError(f"Kritik Hata: Ragas referans bağlam sütununu üretmedi! Mevcut sütunlar: {df.columns.tolist()}")

    df['n_context_chunks'] = df[ref_col].apply(lambda x: len(x) if isinstance(x, list) else 1)
    
    output_file = os.path.join(OUTPUT_DIR, f"testset_mixed_{len(df)}_soru_{timestamp}.csv")
    df.to_csv(output_file, index=False, encoding="utf-8")
    
    chunk_map_df = pd.DataFrame([
        {"chunk_id": d.metadata["chunk_id"], "filename": d.metadata["filename"], "text": d.page_content}
        for d in docs
    ])
    chunk_map_file = os.path.join(OUTPUT_DIR, f"chunk_map_{timestamp}.csv")
    chunk_map_df.to_csv(chunk_map_file, index=False, encoding="utf-8")
    
    # --- KALICI RAPORLAMA DOSYASI ---
    report_lines = []
    report_lines.append("="*60)
    report_lines.append(f"       RAGAS VERİ ÜRETİM RAPORU - {timestamp}")
    report_lines.append("="*60)
    report_lines.append(f"[*] Hedeflenen Soru Sayısı: {TESTSET_SIZE}")
    report_lines.append(f"[*] Gerçekleşen Soru Sayısı: {len(df)}")
    report_lines.append(f"[*] Üretilen Mevcut Sütunlar: {df.columns.tolist()}\n")
    
    if len(df) < TESTSET_SIZE:
        report_lines.append(f"⚠️ BİLGİ: Sistem {TESTSET_SIZE} soru hedeflerken {len(df)} soru üretebildi.\n")
        
    report_lines.append(f"[✓] Test Seti (Soru-Cevap) Kaydedildi: {output_file}")
    report_lines.append(f"[✓] Chunk-ID Haritası Kaydedildi: {chunk_map_file}\n")
    
    synth_col = "synthesizer_name" if "synthesizer_name" in df.columns else "synthesizer_type" if "synthesizer_type" in df.columns else None
    
    report_lines.append("-" * 60)
    report_lines.append("📊 Synthesizer Tipine Göre 'n_context_chunks' Dağılımı:")
    if synth_col:
        report_lines.append(df.groupby(synth_col)['n_context_chunks'].describe().round(2).to_string())
    else:
        report_lines.append(df['n_context_chunks'].describe().round(2).to_string())
        
    report_lines.append("\n" + "-" * 60)
    report_lines.append("[*] Synthesizer Tipine Göre KESİN Exact Match Oranları:")
    doc_texts = {d.page_content for d in docs}
    
    if synth_col:
        for stype in df[synth_col].unique():
            sub = df[df[synth_col] == stype]
            match_count, total_count = 0, 0
            
            for _, row in sub.iterrows():
                contexts = row[ref_col] if isinstance(row[ref_col], list) else [row[ref_col]]
                for ctx in contexts:
                    total_count += 1
                    if ctx in doc_texts:
                        match_count += 1
                        
            match_ratio = (match_count / total_count * 100) if total_count > 0 else 0
            report_lines.append(f"    - [{stype}] Eşleşme: {match_count}/{total_count} ({match_ratio:.1f}%)")
    
    report_lines.append("="*60)
    
    report_text = "\n".join(report_lines)
    print(report_text)
    
    report_file = os.path.join(OUTPUT_DIR, f"generation_report_{len(df)}_soru_{timestamp}.txt")
    with open(report_file, "w", encoding="utf-8") as f:
        f.write(report_text)
        
    print(f"\n[✓] Üretim Raporu Başarıyla Kaydedildi: {report_file}")
    
except Exception as e:
    print(f"\n🔴 RAGAS Üretim Hatası: {e}")